In [1]:
pip install -U langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.14
    Uninstalling langgraph-sdk-0.3.14:
      Successfully uninstalled langgraph-sdk-0.3.14
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.1
    Uninstalling langgraph-1.2.1:
      Successfully uninstalled langgraph-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.1
    Uninstalling langchain-1.3.1:
      Successfully uninstalled langchain-1.3.1


In [2]:
pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.0 MB/s eta 0:00:00


In [24]:
from typing import TypedDict, Annotated
import operator

from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

In [25]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("apikey")
)

In [26]:
class SoftwareAgencyState(TypedDict):
    messages: Annotated[list, operator.add]

    user_requirement: str
    project_type: str

    architecture_plan: str
    backend_plan: str
    database_plan: str
    qa_plan: str

    review_report: str
    final_report: str

In [27]:
def requirement_analyzer(state):

    prompt = f"""
    Analyze the following software requirement:

    {state['user_requirement']}

    Extract:
    - Project Name
    - Features
    - Users
    - Complexity
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [28]:
def project_classifier(state):

    req = state["user_requirement"].lower()

    if any(word in req for word in
           ["ai", "machine learning", "llm", "rag"]):
        ptype = "AI"

    elif any(word in req for word in
             ["website", "portfolio", "ecommerce"]):
        ptype = "Web"

    elif any(word in req for word in
             ["erp", "hospital", "bank", "university", "college"]):
        ptype = "Enterprise"

    else:
        ptype = "Web"

    return {
        "project_type": ptype
    }

In [29]:
def route_project(state):

    if state["project_type"] == "AI":
        return "ai_architect"

    elif state["project_type"] == "Web":
        return "frontend_architect"

    else:
        return "enterprise_architect"

In [30]:
def ai_architect(state):

    prompt = f"""
    Create an AI system architecture for:

    {state['user_requirement']}

    Include:
    - AI Components
    - LLM
    - Vector Database
    - Deployment
    - Tech Stack
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [31]:
def frontend_architect(state):

    prompt = f"""
    Create a frontend architecture for:

    {state['user_requirement']}

    Include:
    - UI Components
    - Routing
    - Pages
    - Tech Stack
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [32]:
def enterprise_architect(state):

    prompt = f"""
    Create an enterprise architecture for:

    {state['user_requirement']}

    Include:
    - Modules
    - User Roles
    - Security
    - Integrations
    - Tech Stack
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [33]:
def backend_agent(state):

    prompt = f"""
    Generate backend plan for:

    {state['user_requirement']}

    Architecture:
    {state['architecture_plan']}

    Include:
    - APIs
    - Authentication
    - Folder Structure
    - Services
    """

    response = llm.invoke(prompt)

    return {
        "backend_plan": response.content
    }

In [34]:
def database_agent(state):

    prompt = f"""
    Generate database design for:

    {state['user_requirement']}

    Include:
    - Tables
    - Relationships
    - Primary Keys
    - Foreign Keys
    - Indexes
    """

    response = llm.invoke(prompt)

    return {
        "database_plan": response.content
    }

In [35]:
def qa_agent(state):

    prompt = f"""
    Generate QA strategy for:

    {state['user_requirement']}

    Include:
    - Test Cases
    - Edge Cases
    - Validation Cases
    """

    response = llm.invoke(prompt)

    return {
        "qa_plan": response.content
    }

In [36]:
def reviewer_agent(state):

    prompt = f"""
    Review the following plans:

    Architecture:
    {state['architecture_plan']}

    Backend:
    {state['backend_plan']}

    Database:
    {state['database_plan']}

    QA:
    {state['qa_plan']}

    Return only:
    APPROVED

    or

    NEEDS_REVISION
    """

    response = llm.invoke(prompt)

    return {
        "review_report": response.content
    }

In [37]:
def final_report_agent(state):

    report = f"""
PROJECT TYPE:
{state['project_type']}

====================================

ARCHITECTURE PLAN:

{state['architecture_plan']}

====================================

BACKEND PLAN:

{state['backend_plan']}

====================================

DATABASE PLAN:

{state['database_plan']}

====================================

QA PLAN:

{state['qa_plan']}

====================================

REVIEW RESULT:

{state['review_report']}
"""

    return {
        "final_report": report
    }

In [38]:
workflow = StateGraph(SoftwareAgencyState)

workflow.add_node("requirement_analyzer", requirement_analyzer)
workflow.add_node("classifier", project_classifier)

workflow.add_node("ai_architect", ai_architect)
workflow.add_node("frontend_architect", frontend_architect)
workflow.add_node("enterprise_architect", enterprise_architect)

workflow.add_node("backend", backend_agent)
workflow.add_node("database", database_agent)
workflow.add_node("qa", qa_agent)
workflow.add_node("reviewer", reviewer_agent)
workflow.add_node("final_report", final_report_agent)

workflow.set_entry_point("requirement_analyzer")

workflow.add_edge("requirement_analyzer", "classifier")

workflow.add_conditional_edges(
    "classifier",
    route_project,
    {
        "ai_architect": "ai_architect",
        "frontend_architect": "frontend_architect",
        "enterprise_architect": "enterprise_architect"
    }
)

workflow.add_edge("ai_architect", "backend")
workflow.add_edge("frontend_architect", "backend")
workflow.add_edge("enterprise_architect", "backend")

workflow.add_edge("backend", "database")
workflow.add_edge("database", "qa")
workflow.add_edge("qa", "reviewer")
workflow.add_edge("reviewer", "final_report")
workflow.add_edge("final_report", END)

graph = workflow.compile()

In [39]:
requirement = input("Enter Software Requirement: ")

result = graph.invoke(
    {
        "messages": [],
        "user_requirement": requirement,
        "project_type": "",
        "architecture_plan": "",
        "backend_plan": "",
        "database_plan": "",
        "qa_plan": "",
        "review_report": "",
        "final_report": ""
    }
)

print(result["final_report"])

Enter Software Requirement: Build a College ERP

PROJECT TYPE:
Enterprise


ARCHITECTURE PLAN:

This document outlines an enterprise architecture for a College ERP (Enterprise Resource Planning) system, designed to streamline administrative, academic, and operational processes within a higher education institution.

---

## Enterprise Architecture for a College ERP System

### I. Executive Summary

The College ERP system aims to be a comprehensive, integrated platform that centralizes data and automates key processes across various departments. It will enhance operational efficiency, improve data accuracy, foster better decision-making, and provide a superior experience for students, faculty, and staff. The architecture emphasizes scalability, security, user-centric design, and seamless integration with existing and future systems.

### II. Business Architecture

#### A. Modules

The ERP will be structured into several interconnected modules, each addressing specific functional areas o